# examining stay ids left after cohort and unit filtering
## seeing some discrepancies between data processed in data_processor_for_shards.py and dynamic_deephit_predictions (trained on the full dataset)

# I have started restricting datasets to sepsis stays (this was already being done for rl but not for dynamic deephit), PICU ward (i.e. excluding NICU), and 

In [1]:
import pandas as pd
import numpy as np
import h5py
import os

In [2]:
# cohort paths
pediatric_age_cohort = '/home/jake/Documents/AI_Clinician/choc_data/cohorts/age/pediatric_stay_ids.csv'

sepsis_cohort = '/home/jake/Documents/AI_Clinician/choc_data/cohorts/sepsis/stay_ids.csv'

picu_cohort = '/home/jake/Documents/AI_Clinician/choc_data/cohorts/icu_units/picu_stay_ids.csv'

# load data
pediatric_age_df = pd.read_csv(pediatric_age_cohort)
sepsis_df = pd.read_csv(sepsis_cohort)
picu_df = pd.read_csv(picu_cohort)

In [3]:
training_full_set_path = '/home/jake/Documents/AI_Clinician/choc_data/combined_data/hourly_all_patients/split_0/training_set/train_dynamic_deephit_dataset.h5'

training_restricted_set_path = '/home/jake/Documents/AI_Clinician/choc_data/combined_data/hourly_all_patients/split_0/training_set/train_sepsis_picu_dynamic_deephit_dataset.h5'

prediction_full_set_path = '/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/final_model/choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260326_040032_999109_job2024209.pbs-7/train_predictions.h5'

prediction_restricted_set_path = '/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/sepsis_picu/final_model/local_39ae437c/train_predictions.h5'

In [4]:
# load trajectories from training_full_set_path and prediction_full_set_path
with h5py.File(training_full_set_path, 'r') as f:
    training_full_dataset = {}
    training_full_dataset['stay_ids'] = f['stay_ids'][:]
    training_full_dataset['trajectories'] = f['trajectories'][:]
    training_full_dataset['index_ptr'] = f['index_ptr'][:]

with h5py.File(prediction_full_set_path, 'r') as f:
    prediction_full_dataset = {}
    prediction_full_dataset['stay_ids'] = f['stay_ids'][:]
    prediction_full_dataset['trajectories'] = f['predictions'][:]
    prediction_full_dataset['index_ptr'] = f['index_ptr'][:]

with h5py.File(training_restricted_set_path, 'r') as f:
    training_restricted_dataset = {}
    training_restricted_dataset['stay_ids'] = f['stay_ids'][:]
    training_restricted_dataset['trajectories'] = f['trajectories'][:]
    training_restricted_dataset['index_ptr'] = f['index_ptr'][:]

with h5py.File(prediction_restricted_set_path, 'r') as f:
    prediction_restricted_dataset = {}
    prediction_restricted_dataset['stay_ids'] = f['stay_ids'][:]
    prediction_restricted_dataset['trajectories'] = f['predictions'][:]
    prediction_restricted_dataset['index_ptr'] = f['index_ptr'][:]

In [5]:
print(f"Training full dataset: {len(training_full_dataset['stay_ids'])} trajectories")
print(f"Prediction full dataset: {len(prediction_full_dataset['stay_ids'])} trajectories")
print(f"Training restricted dataset: {len(training_restricted_dataset['stay_ids'])} trajectories")
print(f"Prediction restricted dataset: {len(prediction_restricted_dataset['stay_ids'])} trajectories")

Training full dataset: 6733 trajectories
Prediction full dataset: 6733 trajectories
Training restricted dataset: 1028 trajectories
Prediction restricted dataset: 1028 trajectories


In [45]:
def filter_by_stay_ids(dataset, stay_ids_to_keep,):

    stay_ids = dataset['stay_ids']
    print(f"Original number of stays: {len(stay_ids)}")
    
    # Create boolean mask for stays to keep
    mask = np.isin(stay_ids, list(stay_ids_to_keep))
    indices_to_keep = np.where(mask)[0]
    
    # logger.info(f"Number of stays after filtering to cohort {cohort}: {len(indices_to_keep)}")
    print(f"Number of stays after filtering by stay ids: {len(indices_to_keep)}")

    # Build trajectory mask
    traj_mask = np.zeros(dataset['trajectories'].shape[0], dtype=bool)
    print(f"Original number of timesteps: {traj_mask.shape[0]}")

    for idx in indices_to_keep:
        start_idx = dataset['index_ptr'][idx]
        end_idx = dataset['index_ptr'][idx + 1]
        traj_mask[start_idx:end_idx] = True

    # Apply masks to all arrays at once
    dataset['stay_ids'] = dataset['stay_ids'][mask]    
    dataset['trajectories'] = dataset['trajectories'][traj_mask]
    
    # Rebuild index_ptr
    new_index_ptr = np.zeros(len(indices_to_keep) + 1, dtype=np.int32)
    for i, idx in enumerate(indices_to_keep):
        start_idx = dataset['index_ptr'][idx]
        end_idx = dataset['index_ptr'][idx + 1]
        new_index_ptr[i + 1] = new_index_ptr[i] + (end_idx - start_idx)
    
    dataset['index_ptr'] = new_index_ptr

    print(f"Number of timesteps after filtering by stay ids: {dataset['trajectories'].shape[0]}")

    return dataset

In [46]:
print(f"training_full_stay_trajectories shape: {training_full_dataset['trajectories'].shape}")
print(f"prediction_full_stay_trajectories shape: {prediction_full_dataset['trajectories'].shape}")

# check that stay_ids are identical across training and prediction sets
training_full_stay_ids_set = set(training_full_dataset['stay_ids'])
prediction_full_stay_ids_set = set(prediction_full_dataset['stay_ids'])

if training_full_stay_ids_set == prediction_full_stay_ids_set:
    print("Stay IDs are identical across training and prediction sets.")
else:
    print("Stay IDs differ between training and prediction sets.")

training_full_stay_trajectories shape: (2488216, 155)
prediction_full_stay_trajectories shape: (2488216, 2, 8)
Stay IDs are identical across training and prediction sets.


In [ ]:
# filter the training_full_stay_ids 
print(f"Number of stay_ids in training_full_stay_ids: {len(training_full_dataset['stay_ids'])}")

# get trajectories longer than 23 hours
trajectory_lengths = training_full_dataset['index_ptr'][1:] - training_full_dataset['index_ptr'][:-1]
long_trajectories = training_full_dataset['stay_ids'][trajectory_lengths > 23]
print(f"Number of stay_ids with trajectories longer than 23: {len(long_trajectories)}")



Number of stay_ids in training_full_stay_ids: 6733
Number of stay_ids with trajectories longer than 71: 6702


In [38]:
# filter the training_full_stay_ids 
print(f"Number of stay_ids in training_full_stay_ids: {len(training_full_dataset['stay_ids'])}")

# filter to sepsis cohort
sepsis_dataset = filter_by_stay_ids(training_full_dataset, sepsis_df['stay_id'].values)
print(f"Number of stay_ids in training_full_stay_ids that are in sepsis cohort: {len(sepsis_dataset['stay_ids'])}")

# filter to only include those that are in picu unit
picu_dataset = filter_by_stay_ids(sepsis_dataset, picu_df['stay_id'].values)
print(f"Number of stay_ids in training_full_stay_ids that are in PICU: {len(picu_dataset['stay_ids'])}")

# filter to only include those that are in pediatric age cohort
pediatric_dataset = filter_by_stay_ids(picu_dataset, pediatric_age_df['stay_id'].values)
print(f"Number of stay_ids in training_full_stay_ids that are in pediatric age cohort: {len(pediatric_dataset['stay_ids'])}")

# get trajectory lengths
trajectory_lengths = np.diff(pediatric_dataset['index_ptr'])
# find stay_id with trajectory less than 1440
short_trajectories = pediatric_dataset['stay_ids'][trajectory_lengths < 1440]
print(f"Number of stay_ids with trajectories less than 1440: {len(short_trajectories)}")
refined_dataset = filter_by_stay_ids(pediatric_dataset, short_trajectories)
print(f"Number of stay_ids in training_full_stay_ids that are in pediatric age cohort and have trajectories less than 1440: {len(refined_dataset['stay_ids'])}")

# get trajectories longer than 71 hours
long_trajectories = pediatric_dataset['stay_ids'][trajectory_lengths > 71]
print(f"Number of stay_ids with trajectories longer than 71: {len(long_trajectories)}")
further_refined_dataset = filter_by_stay_ids(pediatric_dataset, long_trajectories)
print(f"Number of stay_ids in training_full_stay_ids that are in pediatric age cohort and have trajectories longer than 23: {len(further_refined_dataset['stay_ids'])}")


Number of stay_ids in training_full_stay_ids: 6733
Original number of stays: 6733
Number of stays after filtering by stay ids: 1488
Original number of timesteps: 2488216
Number of timesteps after filtering by stay ids: 715424
Number of stay_ids in training_full_stay_ids that are in sepsis cohort: 1488
Original number of stays: 1488
Number of stays after filtering by stay ids: 1111
Original number of timesteps: 715424
Number of timesteps after filtering by stay ids: 457711
Number of stay_ids in training_full_stay_ids that are in PICU: 1111
Original number of stays: 1111
Number of stays after filtering by stay ids: 1036
Original number of timesteps: 457711
Number of timesteps after filtering by stay ids: 424543
Number of stay_ids in training_full_stay_ids that are in pediatric age cohort: 1036
Number of stay_ids with trajectories less than 1440: 1036
Original number of stays: 1036
Number of stays after filtering by stay ids: 1036
Original number of timesteps: 424543
Number of timesteps a